# Notebook 5 · Adaptive KUN for insect-trajectory OOD

**Goal.** Notebook 4 trained a KUN to forecast ONE insect; we reuse its dataset and split verbatim
(same `insect_move`, same `make_dataset` / `make_loaders`). Here we study what happens when the test
insect's motion drifts **out of distribution (OOD)**.

**OOD as a region of parameter space.** Each insect is a point in a space of four generative factors:

1. **Phase** `(phi_x, phi_y)` — shift each channel in time *(type 1)*
2. **Amplitude** `(Ax, Ay)` — per-channel amplitude *(type 2)*
3. **Mixing weights** `(alpha_x, alpha_y)` — how far the amplitude-mixing weights move toward another
   insect *(type 3)*
4. **Frequency** `(fx, fy)` — per-channel frequency scale *(type 4)*

We choose a **training box** in this space — that box is the **in-distribution** region. The
**adaptive KUN** is the notebook-4 KUN trained over that whole box (not a single insect), so it
*adapts to a region* of insect behaviours. Everything outside the box is the **OOD test** region. For
each factor we sweep a 2-D grid and draw a **heat-map of forecast error**: low inside the box, and we
watch how fast it climbs as the insect leaves it.

> **Status: TO BE COMPLETED.** The notebook-4 data + split, the KUN backbone and the plotting
> harness are given. Two `TODO` blocks are yours: **(A)** the **box-randomised training set** (the
> region the adaptive KUN learns) and **(B)** the **heat-map evaluation** over parameter space. Fill
> them and everything runs. Reference:
> [`05_adaptive_forecasting_solution.ipynb`](05_adaptive_forecasting_solution.ipynb).

## Notebook structure

Run top to bottom. **§1** parameters + the **training box**; **§2** the notebook-4 insect,
parameterised by the four factors; **§3** the notebook-4 split, the box-randomised training set, and
**ID-vs-OOD visualisations** of the raw trajectories (separate panels); **§4** the KUN backbone;
**§5** trains the adaptive KUN on the box; **§6** a **univariate** OOD study (one channel at a time,
two curves per factor); **§7** the **bivariate** study — parameter-space **heat-maps** (box outlined);
**§8** the in-box vs. out-of-box table; **§9** **ID-vs-OOD forecasts** (shown separately); **§10**
conclusion.

> Metric: **MSE** on the standardised series (as in notebook 4).

In [ ]:
import random
from math import prod, sin, cos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)
print('Using device:', device)

## 1. Parameters and the training box

Notebook-4 settings, plus the **box**: the in-distribution range of each factor. Each box is **centred
on the ID value**, so the training insects sit in the middle of every range and the OOD test sweeps
out to **both sides**. The KUN trains only inside the box.

In [ ]:
# ---- notebook-4 settings -------------------------------------------
delta_t      = 0.01
features_len = 2                 # channels C : (x, y)
sequence_len = 32                # lookback L
predict_len  = 32                # horizon  H  (this simple 2-layer KUN assumes H == L)
L, H = sequence_len, predict_len

train_len = 1600
test_len  = 400
batch_size = 32
lr        = 3e-4
epochs    = 60

# ---- the four factors: ID value and the TRAINING BOX (in-distribution region) ----
# Each box is CENTRED on the ID value, so OOD lies on BOTH sides (below and above the box).
ID_SEED, ALT_SEED = 16, 777      # the base insect and a 'far' insect (target of the weight axis)
BOX = {
    'amp':   (0.80,  1.20),      # per-channel amplitude   (ID = 1.0, box centre)
    'phase': (-0.70, 0.70),      # per-channel phase       (ID = 0.0, box centre)
    'freq':  (0.93,  1.07),      # per-channel frequency   (ID = 1.0, box centre)
    'alpha': (-0.30, 0.30),      # weight-interpolation    (ID = 0.0, box centre)
}

## 2. The notebook-4 insect, parameterised by the four factors

`insect_init` / `insect_move` are the notebook-4 generator. `insect_move` gains four optional knobs —
phase, amplitude, **per-channel** frequency, and the mixing weights — so one function spans the whole
space; with every knob at its default it reproduces the notebook-4 insect exactly. `insect_at(...)`
builds a trajectory at one point of the space, with `alpha` interpolating the mixing weights from the
ID insect toward a fixed 'far' insect.

In [ ]:
def insect_init(s=122):
    if s > 0:
        random.seed(s)
    insect_init.params_x = [random.gauss(0., 1.) for _ in range(8)]
    insect_init.params_y = [random.gauss(0., 1.) for _ in range(8)]


def make_insect(seed):
    """One insect's 8+8 parameters: 4 amplitudes + 4 frequency offsets per channel."""
    insect_init(seed)
    return list(insect_init.params_x), list(insect_init.params_y)


def insect_move(t, px, py, phase=(0., 0.), amp=(1., 1.), freq=(1., 1.)):
    """Notebook-4 insect with four OOD knobs (phase, amp, per-channel freq; weights via px/py)."""
    [ax1, ax2, ax3, ax4, kx1, kx2, kx3, kx4] = px
    [ay1, ay2, ay3, ay4, ky1, ky2, ky3, ky4] = py
    phx, phy = phase
    fx,  fy  = freq
    x = (ax1*sin(t*(kx1+20)*fx+phx) + ax2*cos(t*(kx2+10)*fx+phx)
         + ax3*sin(t*(kx3+5)*fx+phx) + ax4*cos(t*(kx4+5)*fx+phx)) * amp[0]
    y = (ay1*cos(t*(ky1+20)*fy+phy) + ay2*sin(t*(ky2+10)*fy+phy)
         + ay3*cos(t*(ky3+5)*fy+phy) + ay4*sin(t*(ky4+5)*fy+phy)) * amp[1]
    return x, y


PX,  PY  = make_insect(ID_SEED)        # the in-distribution insect
PX2, PY2 = make_insect(ALT_SEED)       # the 'far' insect (target of the weight axis)


def insect_at(n, amp=(1., 1.), phase=(0., 0.), freq=(1., 1.), alpha=(0., 0.)):
    """(n, 2) trajectory at one point of the parameter space. `alpha` interpolates the four
    amplitude-mixing weights from the ID insect (alpha=0) toward the far insect (alpha=1); the
    frequency offsets stay at the ID insect, so the weight axis is purely 'mixing weights'."""
    ax = [(1-alpha[0])*PX[i] + alpha[0]*PX2[i] if i < 4 else PX[i] for i in range(8)]
    ay = [(1-alpha[1])*PY[i] + alpha[1]*PY2[i] if i < 4 else PY[i] for i in range(8)]
    return np.array([insect_move(t, ax, ay, phase=phase, amp=amp, freq=freq)
                     for t in np.arange(0., n*delta_t, delta_t)][:n], dtype='float32')

## 3. The notebook-4 split + the box-randomised training set

`make_dataset` (chronological split, standardised on **train** stats only) and `make_loaders`
(sliding `L -> H` windows) come from notebook 4. We build the ID train/test loaders, keep the train
**mean / std** (so every other trajectory is standardised the same way), expose `point_loader(...)`
for any point of the space, and build the **box-randomised training set**: several long trajectories
of insects drawn from the box, each windowed like notebook 4. That set is the region the adaptive KUN
learns.

In [ ]:
def make_dataset(dataset, train_len, test_len, L, H):
    """Notebook-4 split: chronological, standardised on TRAIN stats only. Returns the standardised
    (train_set, test_set) AND the train mean/std (so OOD trajectories standardise the same way)."""
    need = train_len + L + H + test_len + L + H
    data = np.array(dataset[:need], dtype='float32')
    train_set = data[:train_len + L + H - 1]
    test_set  = data[train_len + L + H - 1 : train_len + L + H + test_len + L + H - 2]
    mean, std = train_set.mean(0), train_set.std(0)
    return (train_set - mean) / std, (test_set - mean) / std, mean, std


def _windows(data):
    X, Y = [], []
    for i in range(len(data) - L - H + 1):
        X.append(data[i:i+L]); Y.append(data[i+L:i+L+H])
    return torch.from_numpy(np.array(X, 'float32')), torch.from_numpy(np.array(Y, 'float32'))


def make_loaders(train_set, test_set, L, H, bs=batch_size):
    tr = DataLoader(TensorDataset(*_windows(train_set)), batch_size=bs, shuffle=True)
    te = DataLoader(TensorDataset(*_windows(test_set)),  batch_size=bs, shuffle=False)
    return tr, te


# ---- ID train/test (the notebook-4 dataset) + the train statistics ----
ID_dataset = insect_at(train_len + L + H + test_len + L + H + 10)
_id_train, id_test_set, MEAN, STD = make_dataset(ID_dataset, train_len, test_len, L, H)
id_test_loader = make_loaders(_id_train, id_test_set, L, H)[1]


def point_loader(n=160, **knobs):
    """Standardised (by ID train stats) test loader for ONE point of the parameter space."""
    series = (insect_at(n + L + H, **knobs) - MEAN) / STD
    return DataLoader(TensorDataset(*_windows(series)), batch_size=batch_size, shuffle=False)


def box_loader(box, n_traj=16, traj_len=600, seed=7):
    """Training region: long trajectories of insects sampled uniformly from the box, windowed.

    TODO (A): for each of n_traj trajectories, sample every factor uniformly from its box range
        amp   = (U(*box['amp']),   U(*box['amp'])),  and likewise phase / freq / alpha,
      build  traj = (insect_at(traj_len, amp=amp, phase=phase, freq=freq, alpha=alpha) - MEAN) / STD,
      and slide L->H windows over it (step 2) into RX, RY. Return a shuffled DataLoader.
      (Long trajectories + sliding windows let the model see every phase, not just t=0.)
    """
    rng = np.random.default_rng(seed)
    RX, RY = [], []
    # >>> YOUR CODE HERE <<<
    raise NotImplementedError('TODO (A): build the box-randomised training region')


train_loader = box_loader(BOX)
print('box-randomised windows:', len(train_loader.dataset))

In [ ]:
# A look at the space: the ID insect, two in-box insects, and three far-OOD insects.
fig, axes = plt.subplots(1, 6, figsize=(19, 3.4))
shows = [('ID', {}),
         ('in-box #1', dict(amp=(1.15,0.9), phase=(0.5,-0.3), freq=(1.05,0.96))),
         ('in-box #2', dict(alpha=(0.2,-0.15), amp=(0.9,1.15))),
         ('OOD freq',  dict(freq=(1.28,1.28))),
         ('OOD weight',dict(alpha=(0.9,-0.9))),
         ('OOD amp',   dict(amp=(1.5,0.6)))]
for ax, (name, kw) in zip(axes, shows):
    s = insect_at(200, **kw)
    ax.plot(s[:, 0], s[:, 1], lw=1); ax.set_title(name); ax.grid(True)
plt.tight_layout(); plt.show()

### ID and OOD, side by side (separate panels)

ID and each OOD insect are drawn in their **own** panels — the ID column on the left, then one column
per OOD type. Top row is the x-y plane; bottom row is the x channel over time. Axes are shared along
each row, so the panels are directly comparable: amplitude rescales the curve, phase slides it in
time, frequency squeezes it, weight reshapes it.

In [ ]:
VIEW = {'ID': {},
        'OOD amplitude': dict(amp=(1.5, 0.6)),
        'OOD phase':     dict(phase=(1.6, -1.6)),
        'OOD weight':    dict(alpha=(0.9, -0.9)),
        'OOD frequency': dict(freq=(1.28, 1.28))}
n_show = 160
tt = np.arange(n_show) * delta_t
fig, axes = plt.subplots(2, len(VIEW), figsize=(20, 7), sharex='row', sharey='row')
for col, (name, kw) in enumerate(VIEW.items()):
    s = insect_at(n_show, **kw)
    color = 'tab:blue' if name == 'ID' else 'tab:red'
    axes[0, col].plot(s[:, 0], s[:, 1], lw=1.2, c=color)
    axes[0, col].set_title(name); axes[0, col].set_xlabel('x'); axes[0, col].grid(True)
    axes[1, col].plot(tt, s[:, 0], lw=1.2, c=color)
    axes[1, col].set_xlabel('t'); axes[1, col].grid(True)
axes[0, 0].set_ylabel('y  (x-y plane)'); axes[1, 0].set_ylabel('x(t)  (time view)')
fig.suptitle('ID (blue) vs each OOD type (red), shown separately — top: x-y plane, bottom: x vs time',
             y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## 4. The KUN backbone (from notebook 4)

The pluggable **kernel** and the simple **2-layer Kernel U-Net** from notebook 4 — split the length
into patches, run a shared kernel per patch, mirror with U-Net skips. This is the model; the only
thing that makes it *adaptive* is **what we train it on** (Section 5).

In [ ]:
class LinearKernel(nn.Module):
    def __init__(self, ish, osh, idim, odim, **kw):
        super().__init__(); self.ish, self.osh, self.idim, self.odim = tuple(ish), tuple(osh), idim, odim
        self.fc = nn.Linear(prod(ish) * idim, prod(osh) * odim)
    def forward(self, x):
        return self.fc(x.reshape(-1, prod(self.ish) * self.idim)).reshape(-1, *self.osh, self.odim)


class KernelWrapper(nn.Module):
    """Spatial reshape + U-Net skip (kun_lib_v2.py interface)."""
    def __init__(self, ish, osh, idim, odim, mode='encode', unet_skip=True):
        super().__init__(); self.ish, self.osh, self.idim, self.odim = tuple(ish), tuple(osh), idim, odim
        self.mode, self.unet_skip, self._skip = mode, unet_skip, None
        self.kernel = LinearKernel(ish, osh, idim, odim)
    def forward(self, x):
        x = x.reshape(-1, *self.ish, self.idim)
        if self.unet_skip and self.mode == 'encode':
            self._skip = x
        x = self.kernel(x).reshape(-1, *self.osh, self.odim)
        if self.unet_skip and self.mode == 'decode':
            x = x + self._skip
        return x


def two_factors(L):
    a = int(round(L ** 0.5))
    while L % a:
        a -= 1
    return (L // a, a)


class SimpleKUNet(nn.Module):
    """Simple 2-layer Kernel U-Net (notebook 4). Assumes H == L."""
    def __init__(self, L, H, C, hidden=32, latent=64, unet_skip=True):
        super().__init__(); assert H == L
        Lp, P = two_factors(L); self.Lp, self.P, self.C, self.L = Lp, P, C, L
        self.enc1 = KernelWrapper((Lp,), (1,),  C,      hidden, 'encode', unet_skip)
        self.enc2 = KernelWrapper((P,),  (1,),  hidden, latent, 'encode', unet_skip)
        self.dec1 = KernelWrapper((1,),  (P,),  latent, hidden, 'decode', unet_skip)
        self.dec2 = KernelWrapper((1,),  (Lp,), hidden, C,      'decode', unet_skip)
    def forward(self, x):
        B, Lp, P, C = x.shape[0], self.Lp, self.P, self.C
        patches = x.reshape(B, P, Lp, C).reshape(B * P, Lp, C)
        e1 = self.enc1(patches).reshape(B, P, -1)
        z  = self.enc2(e1)
        self.dec1._skip = self.enc2._skip
        self.dec2._skip = self.enc1._skip
        d1 = self.dec1(z).reshape(B * P, 1, -1)
        return self.dec2(d1).reshape(B, self.L, C)

## 5. Train the adaptive KUN on the box

The KUN trains on the box-randomised set — i.e. it adapts to the whole in-distribution **region** of
insect behaviours, not a single insect.

In [ ]:
def mse(model, loader):
    model.eval(); se, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            se += ((model(xb) - yb) ** 2).sum().item(); n += yb.numel()
    return se / n


def train_model(model, loader, epochs):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
    return model


torch.manual_seed(0)
kun = train_model(SimpleKUNet(L, H, features_len), train_loader, epochs)
print('MSE at the ID point: %.3f   |   MSE on the notebook-4 ID test set: %.3f'
      % (mse(kun, point_loader()), mse(kun, id_test_loader)))

## 6. Univariate OOD study — one channel at a time

Each factor has **two** variables: its x-channel and its y-channel (e.g. `(Ax, Ay)`). The simplest
probe varies **one channel at a time**, holding everything else at ID — so every factor gets **two
curves**. The shaded band is the training box (centred on the dashed ID line), so the error is probed
on **both sides**; for the dynamics factors it forms a valley with its floor in the box. This is the
1-D slice the 2-D heat-maps in §7 generalise.

In [ ]:
ID_VAL = {'amp': 1.0, 'phase': 0.0, 'freq': 1.0, 'alpha': 0.0}


def sweep_1d(model, axis, values, channel):
    """MSE as ONE channel of `axis` is swept over `values`, the other channel held at its ID value."""
    iv = ID_VAL[axis]
    out = []
    for v in values:
        pair = (float(v), iv) if channel == 0 else (iv, float(v))
        out.append(mse(model, point_loader(**{axis: pair})))
    return out


SWEEPS = {'amplitude (Ax | Ay)':        ('amp',   np.linspace(0.4,  1.6, 13)),
          'phase (phi_x | phi_y)':      ('phase', np.linspace(-2.0, 2.0, 13)),
          'weight (alpha_x | alpha_y)': ('alpha', np.linspace(-1.0, 1.0, 13)),
          'frequency (fx | fy)':        ('freq',  np.linspace(0.7,  1.3, 13))}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (title, (axis, vals)) in zip(axes, SWEEPS.items()):
    ax.plot(vals, sweep_1d(kun, axis, vals, 0), '-o', ms=3, c='tab:red',    label='vary x-channel')
    ax.plot(vals, sweep_1d(kun, axis, vals, 1), '-s', ms=3, c='tab:purple', label='vary y-channel')
    lo, hi = BOX[axis]
    ax.axvspan(lo, hi, color='tab:green', alpha=0.15, label='training box')
    ax.axvline(ID_VAL[axis], color='gray', ls='--')
    ax.set_title(title); ax.set_xlabel('factor value'); ax.set_ylabel('MSE'); ax.grid(True); ax.legend(fontsize=8)
fig.suptitle('Univariate OOD: vary one channel at a time (two curves per factor)', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## 7. Bivariate OOD study — two channels at once (heat-maps)

Now vary **both** channels of a factor together over a 2-D grid (the other three factors held at ID).
Each cell is the KUN's **MSE** at that point; the white rectangle is the **training box**
(in-distribution), everything outside it is OOD. The univariate curves of §6 are the box's two edges
of these maps.

In [ ]:
def heatmap(model, axis, grid):
    """MSE over a 2-D grid of one factor's (channel-x, channel-y) values.

    TODO (B): fill the grid. For every (a, b) in grid x grid, evaluate the model on a trajectory at
      that point of the plane -- mse(model, point_loader(**{axis: (a, b)})) -- and store it in Z[i, j]
      (i indexes b / rows, j indexes a / cols).
    """
    Z = np.zeros((len(grid), len(grid)))
    # >>> YOUR CODE HERE <<<
    raise NotImplementedError('TODO (B): evaluate the OOD heat-map')
    return Z


PLANES = {
    'amplitude (Ax, Ay)':        dict(axis='amp',   grid=np.linspace(0.4,  1.6, 9)),
    'phase (phi_x, phi_y)':      dict(axis='phase', grid=np.linspace(-2.0, 2.0, 9)),
    'weight (alpha_x, alpha_y)': dict(axis='alpha', grid=np.linspace(-1.0, 1.0, 9)),
    'frequency (fx, fy)':        dict(axis='freq',  grid=np.linspace(0.7,  1.3, 9)),
}
heatmaps = {title: heatmap(kun, p['axis'], p['grid']) for title, p in PLANES.items()}

fig, axes = plt.subplots(1, len(PLANES), figsize=(20, 4.6))
for ax, (title, p) in zip(axes, PLANES.items()):
    grid = p['grid']; lo, hi = BOX[p['axis']]
    Z = heatmaps[title]
    vmax = np.percentile(Z, 80)                       # clip so the in-box structure stays visible
    im = ax.imshow(Z, origin='lower', extent=[grid[0], grid[-1], grid[0], grid[-1]],
                   aspect='auto', cmap='viridis', vmin=0, vmax=max(vmax, 1e-6))
    ax.add_patch(Rectangle((lo, lo), hi - lo, hi - lo, fill=False, ec='white', lw=2))
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Forecast MSE over parameter space  (white box = training / in-distribution region)',
             y=1.03, fontsize=13)
plt.tight_layout(); plt.show()

## 8. In-box vs. out-of-box error

The heat-maps as numbers: mean MSE on points sampled **inside** the box (in-distribution) vs. the
grid cells **outside** it (OOD), per factor.

In [ ]:
def in_box_mse(model, axis, n=24):
    rng = np.random.default_rng(0); lo, hi = BOX[axis]
    return float(np.mean([mse(model, point_loader(**{axis: (float(rng.uniform(lo, hi)),
                                                              float(rng.uniform(lo, hi)))}))
                          for _ in range(n)]))


def out_box_mse(title):
    p = PLANES[title]; grid = p['grid']; lo, hi = BOX[p['axis']]; Z = heatmaps[title]
    out = [Z[i, j] for i, b in enumerate(grid) for j, a in enumerate(grid)
           if not (lo <= a <= hi and lo <= b <= hi)]
    return float(np.mean(out))


summary = pd.DataFrame([{'factor': t, 'in-box (ID)': round(in_box_mse(kun, PLANES[t]['axis']), 3),
                         'out-of-box (OOD)': round(out_box_mse(t), 3)} for t in PLANES]).set_index('factor')
summary

## 9. Forecasts — ID and OOD, shown separately

Two separate figures. **First** the adaptive KUN on the **ID** insect (a few windows): it tracks the
truth. **Then** the **OOD** insects, one panel per factor: the forecast holds up where the KUN absorbs
the shift and breaks where OOD bites (frequency). Each panel: input / true / prediction, with its MSE.

In [ ]:
@torch.no_grad()
def forecast(kw):
    Xo, Yo = (t.numpy() for t in _windows((insect_at(160 + L + H, **kw) - MEAN) / STD))
    s0 = len(Xo) // 2
    pred = kun(torch.from_numpy(Xo[s0:s0+1]).to(device)).cpu().numpy()[0]
    return Xo[s0], Yo[s0], pred


def draw(ax, xin, ytrue, pred, title):
    ax.plot(xin[:, 0],   xin[:, 1],   'o-',  c='tab:blue',  ms=3, label='input')
    ax.plot(ytrue[:, 0], ytrue[:, 1], 'o-',  c='tab:green', ms=3, label='true')
    ax.plot(pred[:, 0],  pred[:, 1],  'x--', c='tab:red',   ms=4, label='pred')
    ax.set_title(title); ax.grid(True)


# ---- Figure 1 : IN-DISTRIBUTION (ID) ----
Xo, Yo = (t.numpy() for t in _windows((insect_at(200 + L + H) - MEAN) / STD))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
for ax, s0 in zip(axes, np.linspace(0, len(Xo) - 1, 3, dtype=int)):
    with torch.no_grad():
        pred = kun(torch.from_numpy(Xo[s0:s0+1]).to(device)).cpu().numpy()[0]
    draw(ax, Xo[s0], Yo[s0], pred, f'ID window {s0}  (MSE {((pred-Yo[s0])**2).mean():.2f})')
axes[0].legend(fontsize=8)
fig.suptitle('In-distribution — the KUN tracks the truth', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Figure 2 : OUT OF DISTRIBUTION (OOD), one panel per factor ----
OOD_FC = {'OOD amplitude (1.5, 0.6)':  dict(amp=(1.5, 0.6)),
          'OOD phase (1.6, -1.6)':     dict(phase=(1.6, -1.6)),
          'OOD weight (0.9, -0.9)':    dict(alpha=(0.9, -0.9)),
          'OOD frequency (1.28, 1.28)': dict(freq=(1.28, 1.28))}
fig, axes = plt.subplots(1, 4, figsize=(18, 4.3))
for ax, (name, kw) in zip(axes, OOD_FC.items()):
    xin, ytrue, pred = forecast(kw)
    draw(ax, xin, ytrue, pred, f'{name}\nMSE {((pred-ytrue)**2).mean():.2f}')
axes[0].legend(fontsize=8)
fig.suptitle('Out of distribution — the forecast degrades where OOD bites', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## What to finish

- [ ] **(A) `box_loader`** — the box-randomised training region (Section 3).
- [ ] **(B) `heatmap`** — evaluate the KUN's MSE over each parameter plane (Section 7; the univariate
  `sweep_1d` in Section 6 is the worked example to copy from).

Once both run, Sections 5-9 train the adaptive KUN and draw the sweeps, heat-maps, table and OOD
forecasts automatically.

## 10. Conclusion

_Fill in from your own run._

**Reading the studies (Sections 6-8).** The box is **centred** on ID, so OOD spreads to **both
sides**:
- **Phase** stays flat across the whole sweep, both sides: the KUN is essentially **phase-invariant**,
  so this OOD type is absorbed for free.
- **Amplitude** is **monotonic**, not symmetric: the small-amplitude side stays cheap (a smaller
  signal is a smaller absolute MSE), while the large-amplitude side warms up — the model tolerates
  the shift either way.
- **Weight** is a clean **U** — moving the mixing weights *either* direction away from the ID insect
  (alpha up or down) raises the error roughly symmetrically.
- **Frequency** is the hard axis and also a **U**, but a steep one: speeding the insect up *or*
  slowing it down both break the forecast, because changing the *dynamics* is the OOD the KUN cannot
  extrapolate through.

**Takeaway.** OOD is a *region* problem: a forecaster is reliable inside the parameter box it was
trained on and degrades on **either side** of it. Training the KUN over a **box centred on ID**
(domain randomisation) — rather than a single insect — is what makes it *adaptive* to that whole
region. The direction that most needs the box widened is **frequency**.

**Reflection:**
- Which factors did the KUN absorb almost for free (flat sweep), and which gave a U-shaped curve with
  OOD on both sides?
- Why is the amplitude curve monotonic rather than U-shaped like weight and frequency?
- Widen `BOX['freq']` and retrain: does the frequency valley get wider? At what cost to in-box
  accuracy or training time?

**Extensions (optional):**
- **Test-time adaptation** — a few self-supervised steps on the latest window before forecasting — to
  chase the frequency axis the static box cannot reach.
- Swap the KUN kernel (`linear` -> `mlp` / `lstm` / `transformer`, as in notebook 4).

See [Kernel U-Net](https://jiangyou2025.github.io/kun/zh/kernel-u-net/).